# Verify Resume Checkpoints

Notebook nay chi dung de verify same-stage resume cho `align_latest.pt` va `finetune_latest.pt`. No khong train full va khong dung checkpoint train that cho align probe.

In [ ]:
GITHUB_REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
GITHUB_BRANCH = "huy"
USE_SAFE_CONFIG = False

# Feature cache repo. De trong neu muon build cache lai.
CACHE_HF_REPO_ID = "huyvanzzz/cache"
CACHE_HF_REPO_TYPE = "dataset"

# Finetune latest checkpoint da upload len HF de verify resume finetune.
FINETUNE_CHECKPOINT_HF_REPO_ID = ""
FINETUNE_CHECKPOINT_HF_REPO_TYPE = "model"
FINETUNE_CHECKPOINT_FILENAME = "finetune_latest.pt"
# Optional: only set True for a checkpoint saved by the current fixed code.
RUN_HF_FINETUNE_CHECKPOINT_VERIFY = False

import os
os.environ["GITHUB_REPO_URL"] = GITHUB_REPO_URL
os.environ["GITHUB_BRANCH"] = GITHUB_BRANCH
os.environ["USE_SAFE_CONFIG"] = "1" if USE_SAFE_CONFIG else "0"
os.environ["CACHE_HF_REPO_ID"] = CACHE_HF_REPO_ID
os.environ["CACHE_HF_REPO_TYPE"] = CACHE_HF_REPO_TYPE
os.environ["FINETUNE_CHECKPOINT_HF_REPO_ID"] = FINETUNE_CHECKPOINT_HF_REPO_ID
os.environ["FINETUNE_CHECKPOINT_HF_REPO_TYPE"] = FINETUNE_CHECKPOINT_HF_REPO_TYPE
os.environ["FINETUNE_CHECKPOINT_FILENAME"] = FINETUNE_CHECKPOINT_FILENAME
os.environ["RUN_HF_FINETUNE_CHECKPOINT_VERIFY"] = "1" if RUN_HF_FINETUNE_CHECKPOINT_VERIFY else "0"
print({
    "repo": GITHUB_REPO_URL,
    "branch": GITHUB_BRANCH,
    "safe_config": USE_SAFE_CONFIG,
    "cache_hf_repo": CACHE_HF_REPO_ID,
    "finetune_checkpoint_hf_repo": FINETUNE_CHECKPOINT_HF_REPO_ID,
    "finetune_checkpoint_filename": FINETUNE_CHECKPOINT_FILENAME,
    "run_hf_finetune_checkpoint_verify": RUN_HF_FINETUNE_CHECKPOINT_VERIFY,
})

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        print("HF_TOKEN secret not loaded:", type(exc).__name__)
print("HF_TOKEN present:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  BASE_CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_safe.yaml
else
  BASE_CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu.yaml
fi
FINETUNE_VERIFY_PROFILE=outputs/resume_verify_finetune_tmp
FINETUNE_VERIFY_CONFIG=/tmp/repvit_t5_efficient_mini_resume_verify_finetune.yaml
export BASE_CONFIG FINETUNE_VERIFY_PROFILE FINETUNE_VERIFY_CONFIG
python - <<'PY'
import os
from pathlib import Path
import yaml
cfg = yaml.safe_load(Path(os.environ['BASE_CONFIG']).read_text())
cfg['project']['output_dir'] = os.environ['FINETUNE_VERIFY_PROFILE']
cfg['training']['align_epochs'] = 1
cfg['training']['finetune_epochs'] = 1
Path(os.environ['FINETUNE_VERIFY_CONFIG']).write_text(yaml.safe_dump(cfg, sort_keys=False))
PY
rm -rf "$FINETUNE_VERIFY_PROFILE"
python -m efficient_vlm_ad prepare-data --config "$FINETUNE_VERIFY_CONFIG" --subset smoke
PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$FINETUNE_VERIFY_CONFIG" --stage align --max-steps 2 --debug --debug-samples 1 --debug-numerics --progress-log-every-steps 1
PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$FINETUNE_VERIFY_CONFIG" --stage finetune --resume "$FINETUNE_VERIFY_PROFILE/checkpoints/align_best.pt" --max-steps 2 --debug --debug-samples 1 --debug-numerics --progress-log-every-steps 1
python -m efficient_vlm_ad verify-resume-checkpoint \
  --config "$FINETUNE_VERIFY_CONFIG" \
  --stage finetune \
  --checkpoint "$FINETUNE_VERIFY_PROFILE/checkpoints/finetune_latest.pt" \
  --debug \
  --debug-samples 1 \
  --debug-numerics \
  --no-progress
cat "$FINETUNE_VERIFY_PROFILE/debug/finetune_resume_checkpoint_verify.json"

In [ ]:
%%bash
set -e
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving/.git ]; then
  git clone --branch "$GITHUB_BRANCH" "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving
fi
cd Efficient_VLM_For_Autonomous_Driving
git fetch origin "$GITHUB_BRANCH"
git checkout "$GITHUB_BRANCH"
git pull --ff-only origin "$GITHUB_BRANCH"
python -m pip install -e .
python -m efficient_vlm_ad --help | head -40

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_safe.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_safe
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu
fi
echo "CONFIG=$CONFIG PROFILE=$PROFILE"
python -m efficient_vlm_ad inspect-data --config "$CONFIG"
python -m efficient_vlm_ad prepare-data --config "$CONFIG" --subset full
if [ -n "$CACHE_HF_REPO_ID" ]; then
  echo "Restoring feature cache from HF: $CACHE_HF_REPO_ID"
  rm -rf "$PROFILE/cache"
  mkdir -p "$PROFILE/cache"
  hf download "$CACHE_HF_REPO_ID" --repo-type "$CACHE_HF_REPO_TYPE" --local-dir "$PROFILE/cache"
else
  echo "CACHE_HF_REPO_ID empty; building cache."
  python -m efficient_vlm_ad prepare-features --config "$CONFIG" --subset full --debug --debug-samples 1
fi
find "$PROFILE/cache" -maxdepth 2 -type f | head -30

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  BASE_CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_safe.yaml
else
  BASE_CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu.yaml
fi
ALIGN_VERIFY_PROFILE=outputs/resume_verify_align_tmp
ALIGN_VERIFY_CONFIG=/tmp/repvit_t5_efficient_mini_resume_verify_align.yaml
export BASE_CONFIG ALIGN_VERIFY_PROFILE ALIGN_VERIFY_CONFIG
python - <<'PY'
import os
from pathlib import Path
import yaml
cfg = yaml.safe_load(Path(os.environ['BASE_CONFIG']).read_text())
cfg['project']['output_dir'] = os.environ['ALIGN_VERIFY_PROFILE']
cfg['training']['align_epochs'] = 1
cfg['training']['finetune_epochs'] = 1
Path(os.environ['ALIGN_VERIFY_CONFIG']).write_text(yaml.safe_dump(cfg, sort_keys=False))
PY
rm -rf "$ALIGN_VERIFY_PROFILE"
python -m efficient_vlm_ad prepare-data --config "$ALIGN_VERIFY_CONFIG" --subset smoke
PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$ALIGN_VERIFY_CONFIG" --stage align --max-steps 2 --debug --debug-samples 1 --debug-numerics --progress-log-every-steps 1
python -m efficient_vlm_ad verify-resume-checkpoint \
  --config "$ALIGN_VERIFY_CONFIG" \
  --stage align \
  --checkpoint "$ALIGN_VERIFY_PROFILE/checkpoints/align_latest.pt" \
  --debug \
  --debug-samples 1 \
  --debug-numerics \
  --no-progress
cat "$ALIGN_VERIFY_PROFILE/debug/align_resume_checkpoint_verify.json"

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$RUN_HF_FINETUNE_CHECKPOINT_VERIFY" != "1" ]; then
  echo "Skipping optional HF finetune checkpoint verify. Use the fresh checkpoint cell above for the standard resume test."
  exit 0
fi
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_safe.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_safe
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu
fi
if [ -z "$FINETUNE_CHECKPOINT_HF_REPO_ID" ]; then
  echo "Set FINETUNE_CHECKPOINT_HF_REPO_ID in the first cell before running this cell."
  exit 1
fi
mkdir -p "$PROFILE/checkpoints"
hf download "$FINETUNE_CHECKPOINT_HF_REPO_ID" "$FINETUNE_CHECKPOINT_FILENAME" --repo-type "$FINETUNE_CHECKPOINT_HF_REPO_TYPE" --local-dir "$PROFILE/checkpoints"
if command -v sha256sum >/dev/null 2>&1; then
  sha256sum "$PROFILE/checkpoints/$FINETUNE_CHECKPOINT_FILENAME"
fi
python -m efficient_vlm_ad verify-resume-checkpoint \
  --config "$CONFIG" \
  --stage finetune \
  --checkpoint "$PROFILE/checkpoints/finetune_latest.pt" \
  --debug \
  --debug-samples 1 \
  --debug-numerics \
  --no-progress
cat "$PROFILE/debug/finetune_resume_checkpoint_verify.json"